In [0]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

# --- Configuration ---
# List of your Delta tables to process
delta_tables_to_clean = [
    "raw.customer",
    "raw.orders",
    "raw.products",
    "processed.orders",
    "processed.customers",
    "processed.products",
    "presentation.ecomm_profit_agg"
]


SHORT_RETENTION_INTERVAL = "interval 24 hour" # For ALTER TABLE properties
SHORT_RETENTION_HOURS = 24                   # For VACUUM command parameter

# --- Warning Messages ---
print("\n" + "="*80)
print("  !!!  WARNING: EXTREMELY AGGRESSIVE DELTA TABLE HISTORY CLEANUP  !!!")
print("  This operation is IRREVERSIBLE and will permanently delete historical data.")
print(f"  Time travel and historical data access beyond {SHORT_RETENTION_INTERVAL} will be LOST.")
print("  Ensure you understand the implications and have backups if necessary.")
print("  !!!  DO NOT RUN ON PRODUCTION WITHOUT CAREFUL CONSIDERATION  !!!")
print("="*80 + "\n")


# --- History Cleanup Logic ---
for table_name in delta_tables_to_clean:
    print(f"\n--- Processing table: {table_name} ---")
    try:
        # Step 1: Set 'delta.logRetentionDuration' property
        spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES ('delta.logRetentionDuration' = '{SHORT_RETENTION_INTERVAL}')")
        print(f"  Set 'delta.logRetentionDuration' for {table_name} to '{SHORT_RETENTION_INTERVAL}'.")

        # Step 2: Set 'delta.deletedFileRetentionDuration' property
        spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '{SHORT_RETENTION_INTERVAL}')")
        print(f"  Set 'delta.deletedFileRetentionDuration' for {table_name} to '{SHORT_RETENTION_INTERVAL}'.")

        # Step 3: Run VACUUM command to physically remove the old files and log entries
        # CORRECTION: Pass SHORT_RETENTION_HOURS as a positional argument.
        print(f"  Running VACUUM on {table_name} with RETAIN {SHORT_RETENTION_HOURS} HOURS...")
        DeltaTable.forName(spark, table_name).vacuum(SHORT_RETENTION_HOURS) # Corrected line
        print(f"  VACUUM complete for {table_name}.")
        print(f"  History for {table_name} has been aggressively cleaned.")

    except Exception as e:
        print(f"  ERROR: Failed to process table {table_name}. Reason: {e}")


print("\n" + "="*80)
print("  One-time Delta table history cleanup process finished.")
print("  Consider resetting 'delta.logRetentionDuration' and 'delta.deletedFileRetentionDuration'")
print("  to your desired longer retention (e.g., 30 days and 7 days respectively) ")
print("  if you want to enable time travel for longer periods in the future.")
print("="*80 + "\n")

# spark.stop() # Uncomment if running as a standalone script outside of a Databricks notebook environment

In [0]:
%sql show partitions presentation.ecomm_profit_agg